In [1]:
import numpy as np
import matplotlib.pylab as plt

In [2]:
import numpy as np

size = 4
rank = 0

arr = np.zeros(5)

indices = np.zeros(size+1)

for i in range(size):
    indices[i] = 5 * rank / size
    print(str(indices[i]) + " ")





0.0 
0.0 
0.0 
0.0 


In [4]:


test = True
for p in range(1,100000):
    q = int(np.ceil(np.log2(p)))
    s = p

    s_arr = np.zeros(q+1, dtype=int)
    for i in range(q+1):
        s_arr[q-i] = s
        s = int(np.ceil(s/2))

    for k in range(q):
        eps = (s_arr[k+1]%2 != 0)
        if (s_arr[k] - eps != s_arr[k+1]-s_arr[k]):
            test = False

print(test)


True


In [5]:
# test for finding correct amount of blocks to send in irregular allgather
def getSendSchedule(p):
    blocks = np.ones(p, dtype=int)

    s = p

    sendSched = []
    shiftSched = []
    # calculate how many blocks should be sent
    while s > 1:
        s_old  = s
        s = int(np.ceil(s/2))
        shiftSched.append(s)

        sendSched.append(blocks[s])
        blocks[:s_old-s]= blocks[:s_old-s] + blocks[s:s_old]

    return sendSched, shiftSched

def getDataArr(p, sendSched):
    s = p
    data = -np.ones((p,p))
    data[0] = np.arange(p)
    sumsend = 1
    while s > 1:
        s_old  = s
        s = int(np.ceil(s/2))


        nsend = sendSched.pop(0)
        for i in range(nsend):
            data[sumsend+i] = np.roll(data[i], -s)
        sumsend += nsend
    return data

def checkData(data):
    p = data.shape[0]
    return np.all(np.sort(data[:,0]) == np.arange(p))


p=31*32
sendSched, shiftSched = getSendSchedule(p)
data = getDataArr(p, sendSched)
check = checkData(data)
check

True

In [6]:

p = 25
s = p
blocks = np.ones(p, dtype=int)

sendSched = []
shiftSched = []
# calculate how many blocks should be sent
while s > 1:
    s_old  = s
    s = int(np.ceil(s/2))

    blocks[0] = blocks[int(np.ceil(s/2))]
    blocks[:s_old-s] = blocks[:s_old-s] + blocks[s:s_old]

    print(eps, blocks[:s]) 



True [2 2 2 2 2 2 2 2 2 2 2 2 1]
True [4 4 4 4 4 3 2]
True [8 7 6 4]
True [13 11]
True [22]


In [7]:
p=10
sendSched, shiftSched = getSendSchedule(p)

data = -np.ones((p,p), dtype=int)
data[0] = np.arange(p)
shiftSched.insert(0,p)
shiftSched
for i in range(len(shiftSched)-1):
    sendSched[i] = shiftSched[i] - shiftSched[i+1]

print(sendSched)
sumsend = 1
s = p
count = 0
nsend = 1
sendNext = sendSched.pop()
while s > 1:
    s_old  = s
    s = int(np.ceil(s/2))


    send = sendNext
    sendNext = sendSched.pop()
    up=send
    print(up)
    low = sendNext%2

    for i in range(low, up):
        if(sumsend+i-low > p-1):
            break
        data[sumsend+i-low] = np.roll(data[i], -s)

    sumsend += up-low
#np.all(np.sort(data[:p,0]) == np.arange(p))
data

[5, 2, 1, 1]
1
1
2


IndexError: pop from empty list

In [ ]:
def nearestPow2(p: int):
    p = int(p)
    count = 0
    while p != 1:
        p = p >>1
        count += 1
    return 1<<count
nearestPow2(25), nearestPow2(16), nearestPow2(32)

(16, 16, 32)

In [1]:
#  testing adapted skip sequence to mostly halfing
checktotal = True
for p in range(2,1000):
    data = -np.ones((p,p), dtype=int)
    data[0] = np.arange(p)
    state = np.ones(p, dtype=int)
    s_old = p
    s = nearestPow2(p)
    nsend=1
    brecv=1
    while s > 0:
        if state[0] != brecv:
            temp = np.copy(data[state[0]-1])
        for i in range(nsend-1, -1, -1):
            data[brecv+i] = np.roll(data[i], -s)
        if state[0] != brecv:
            data[brecv + nsend] = temp
        state[:s_old-s]= state[:s_old-s] + state[s:s_old]
        nsend=state[s//2]
        brecv = state[s//2-1]
        #print(s, state[:s], nsend, brecv)
        s_old = s
        s = s//2
    check = np.all(np.sort(data[:,0])==np.arange(p))
    if not check:
        checktotal =False
print(checktotal)
    

NameError: name 'np' is not defined

In [52]:
for i in range(4-1,-1,-1):
    print(i)

3
2
1
0


In [6]:
setsize = 10000000
p = 32*32
crit_max = setsize // 32
crit_min = setsize // (32 * p)
crit_mean = (setsize * (p-1)) // (32 * p * int(np.log2(p)))

print(crit_max + 10, crit_min + 10, crit_mean + 10)

312510 315 31229


In [22]:
p = 24
tail = p
shift = 1 << int(np.log2(p-1))

tail, shift

inbuff = False

buffswitch = 0

while(shift>=1):
    print(tail, shift)
    if (tail>=shift or tail==0):
        if inbuff:
            buffswitch+=1
        inbuff = False
        if tail !=0:
            tail -= shift
    else:
        if not inbuff:
            buffswitch += 1
        inbuff = True

    shift //= 2
if inbuff:
    buffswitch += 1
print("buffer copies: " + str(buffswitch))

24 16
8 8
0 4
0 2
0 1
buffer copies: 0


In [104]:
np.floor(10**(np.random.uniform(-1, 1, 40)-0.3)) * 10**(np.random.uniform(-1, 1, 40)-0.3)

array([0.        , 0.09424857, 0.        , 0.319381  , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.14181619, 3.39268858, 0.4495244 , 2.4702656 ,
       0.        , 1.05118827, 0.        , 0.13524991, 0.        ,
       0.32126888, 0.        , 0.        , 2.47289629, 6.12750645,
       2.49474728, 0.        , 0.        , 0.1634701 , 0.        ,
       0.        , 0.        , 6.04089478, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ])

In [110]:
# test for count sketch

X = np.array([0.        , 0.09424857, 0.        , 0.319381  , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.14181619, 3.39268858, 0.4495244 , 2.4702656 ,
       0.        , 1.05118827, 0.        , 0.13524991, 0.        ,
       0.32126888, 0.        , 0.        , 2.47289629, 6.12750645,
       2.49474728, 0.        , 0.        , 0.1634701 , 0.        ,
       0.        , 0.        , 6.04089478, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ])
N_x = np.size(X)
N_y = 50
d = 5


indices = np.zeros((N_x, d), dtype=int)
signs = np.zeros((N_x, d), dtype=int)
Y = np.zeros(N_y)

# compress
for i, x in enumerate(X):
    for j in range(d):
        index = hash(i+0.865488553*j)%N_y
        sign= (hash(x+0.365465*j)%2)*2 - 1

        Y[index] += sign * x

        signs[i,j] = sign
        indices[i,j] = index



# decrompress
vals = np.zeros(d)
X_dec = np.zeros(N_x)
for i in range(N_x):
    for j in range(d):
        vals[j] = Y[indices[i,j]]*signs[i,j]
    X_dec[i] = np.median(vals)

print(X)
print(X_dec)



[0.         0.09424857 0.         0.319381   0.         0.
 0.         0.         0.         0.         0.         0.14181619
 3.39268858 0.4495244  2.4702656  0.         1.05118827 0.
 0.13524991 0.         0.32126888 0.         0.         2.47289629
 6.12750645 2.49474728 0.         0.         0.1634701  0.
 0.         0.         6.04089478 0.         0.         0.
 0.         0.         0.         0.        ]
[ 0.00000000e+00  9.42485700e-02  0.00000000e+00  4.54630910e-01
 -9.42485700e-02  0.00000000e+00  0.00000000e+00  0.00000000e+00
 -2.17347840e+00  0.00000000e+00  0.00000000e+00  3.05286290e-01
  2.50497041e+00  4.49524400e-01  2.33501569e+00  0.00000000e+00
  1.05118827e+00  0.00000000e+00  4.10013400e-02  0.00000000e+00
  1.88788000e-03  0.00000000e+00  0.00000000e+00  2.47289629e+00
  5.80623757e+00  2.49474728e+00  0.00000000e+00  0.00000000e+00
 -2.54549095e+00 -4.49524400e-01 -2.02074120e+00 -2.33764638e+00
  6.18271097e+00 -1.05118827e+00  0.00000000e+00  2.33501569e+00